# Banana Lifeline — Training on a Colab T4

Set up for: **notebook edited locally in VS Code, kernel running on a Colab T4.**

The catch with that setup is that the file you are editing lives on your PC, but the code
runs on Google's machine — which cannot see your `E:\` drive. Cell 1 checks this and tells
you where you actually are, so nothing has to be assumed.

This notebook only drives things. The real logic lives in the `banana_lifeline` package,
which is tested and type-checked; keeping it out of notebook cells stops it drifting from
the repo.

## 1. Where is this code actually running?

In [1]:
import os, pathlib, platform, shutil, socket, subprocess, sys

IN_COLAB_VM = pathlib.Path("/content").is_dir() or "google.colab" in sys.modules
HAS_GPU = shutil.which("nvidia-smi") is not None
GPU = ""
if HAS_GPU:
    GPU = subprocess.run(
        "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
        shell=True, capture_output=True, text=True,
    ).stdout.strip()

print(f"os        {platform.system()} {platform.release()}")
print(f"host      {socket.gethostname()}")
print(f"cwd       {pathlib.Path.cwd()}")
print(f"GPU       {GPU or 'NONE'}\n")

if IN_COLAB_VM:
    print(">> Kernel is on a GOOGLE COLAB VM, not your PC.")
    print("   Your local drive is invisible here. Use the Drive zip in step 2.")
else:
    print(">> Kernel is on your OWN machine.")
    print("   Local files work directly. But check the GPU line above —")
    print("   'NONE' means training takes hours, not minutes.")

if not HAS_GPU:
    print("\n!! No GPU visible. If you expected a T4, the kernel is not attached to the")
    print("   Colab GPU runtime. Reconnect it before training.")

os        Linux 6.6.122+
host      dab85e73a178
cwd       /content
GPU       Tesla T4, 15360 MiB

>> Kernel is on a GOOGLE COLAB VM, not your PC.
   Your local drive is invisible here. Use the Drive zip in step 2.


## 2. Get the project onto this machine

Your repo is **public**, and the image cache is committed to it, so a plain `git clone`
brings the code *and* the 2,584 training images in one step. No Drive upload, no token.

If the kernel is already sitting on your own machine, this finds the folder and downloads
nothing.


In [9]:
REPO_URL = "https://github.com/Muhammad-Saaaad/Banana-Lifeline.git"


def find_local_root():
    """Look for the repo on whatever filesystem this kernel can see.

    Searches upward from the working directory, then the usual clone locations. Walking
    up alone is not enough: if the notebook is launched from /content but the clone lands
    in /content/Banana-Lifeline, an upward-only search misses it.
    """
    here = pathlib.Path.cwd().resolve()
    candidates = [here, *here.parents,
                  pathlib.Path("/content/Banana-Lifeline"),
                  here / "Banana-Lifeline"]
    for base in candidates:
        for probe in (base / "ai", base):
            if (probe / "pyproject.toml").is_file() and (probe / "src" / "banana_lifeline").is_dir():
                return probe.resolve()
    return None


AI_ROOT = find_local_root()

if AI_ROOT is not None:
    print("Found the project already on this filesystem — nothing to download.")
else:
    target = pathlib.Path("/content") if IN_COLAB_VM else pathlib.Path.cwd()
    print(f"Cloning into {target} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], cwd=str(target), check=True)
    AI_ROOT = (target / "Banana-Lifeline" / "ai").resolve()

# Every module in the package resolves paths through this one variable.
os.environ["BANANA_AI_ROOT"] = str(AI_ROOT)

n_cached = len(list((AI_ROOT / "data" / "cache").glob("*.jpg")))
print(f"
AI root:       {AI_ROOT}")
print(f"cached images: {n_cached}   (expect 2584)")

if n_cached < 2000:
    print("
!! Cache incomplete. Check that ai/data/cache exists in the clone:")
    print(f"   {AI_ROOT / 'data'}")
    for p in sorted((AI_ROOT / "data").glob("*")):
        n = len(list(p.glob("*"))) if p.is_dir() else "-"
        print(f"     {p.name:20} {n}")
    raise SystemExit("Stop here — training needs the image cache.")


Project not visible here. Fetching from Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

AI root:       /content/project/ai
cached images: 0   (expect 2584)


AssertionError: Image cache missing or incomplete — check the zip upload.

## 3. Install

`--no-deps` stops pip swapping Colab's CUDA-matched torch for a generic build.

In [ ]:
%pip install -q timm albumentations grad-cam scikit-learn
%pip install -q -e "{AI_ROOT}" --no-deps

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    if not torch.cuda.is_bf16_supported():
        print("  -> Turing (T4): fp16 only. train.yaml already sets amp_dtype: float16.")
else:
    print("\nNO GPU — attach the Colab GPU runtime before training.")

## 4. Check the split before spending GPU time

Asserts no photograph appears in two groups. If this fails, **stop** — every number after it
would be meaningless. This is the exact bug the project exists to have fixed, so it is
checked, not assumed.

In [ ]:
import pandas as pd
from banana_lifeline.config import get_paths, load_data_config, load_train_config
from banana_lifeline.data.dataset import load_manifest, manifest_path
from banana_lifeline.data.split import verify_no_leakage

paths, data_cfg, train_cfg = get_paths(), load_data_config(), load_train_config()
clean = load_manifest(manifest_path(paths, "clean"))
verify_no_leakage(clean)

print("Leakage check PASSED.\n")
print(pd.crosstab(clean["label"], clean["split"]))

## 5. Train on the honest split

~10–20 min on a T4. Watch **panama recall**, not accuracy — sigatoka outnumbers panama about
4:1, so accuracy can look healthy while the model ignores the disease that kills the plant.

**On CUDA out-of-memory:** uncomment the override below. Same effective batch, half the memory.

A checkpoint is written every epoch, so a dropped connection costs one epoch, not the run.

In [ ]:
from banana_lifeline.vision.train import train

# train_cfg = train_cfg.model_copy(update={"batch_size": 16, "grad_accum_steps": 2})

print(f"{train_cfg.model.architecture} @ {train_cfg.image_size}px, "
      f"batch {train_cfg.batch_size}x{train_cfg.grad_accum_steps}, amp={train_cfg.amp_dtype}\n")

clean_run = train(clean, train_cfg, data_cfg, paths, run_name="clean")

## 6. Train on the contaminated split

Same recipe, same settings, same excluded photos. **Only the split differs** — that is what
makes the comparison mean something. Expect this one to look great; that is the point, since
it is the number the broken split would have let you report.

In [ ]:
original = load_manifest(manifest_path(paths, "original"))
original_run = train(original, train_cfg, data_cfg, paths, run_name="original")

## 7. Score, calibrate, compare

Calibration is fitted on the **practice** set, never on test — tuning a threshold on test
data leaks it and inflates the result.

In [ ]:
from banana_lifeline.vision.calibrate import calibrate
from banana_lifeline.vision.evaluate import (
    background_ablation, comparison_report, predict, write_evaluation,
)
from banana_lifeline.vision.model import load_checkpoint

loaded    = load_checkpoint(clean_run / "best.pt")
test_rows = clean[clean["split"] == "test"]
val_rows  = clean[clean["split"] == "val"]

clean_out = predict(loaded.model, test_rows, loaded.train_config, data_cfg, paths)
print("HONEST :", clean_out.metrics.headline())

original_loaded = load_checkpoint(original_run / "best.pt")
original_out = predict(
    original_loaded.model,
    original[original["split"] == "test"],
    original_loaded.train_config, data_cfg, paths,
)
print("CLAIMED:", original_out.metrics.headline())

val_out = predict(loaded.model, val_rows, loaded.train_config, data_cfg, paths)
calibration = calibrate(val_out.logits, val_out.labels)
print("\n", calibration.summary())

### Is it reading the leaf, or the background?

Every photo came from one farm on one day, so sky, soil and light barely change between
classes — a model can score well by reading those instead of the plant. Cropping tightly
removes that context. **Small drop** = reading the leaf. **Large drop** = the score will not
survive a different farm.

In [ ]:
ablation = background_ablation(loaded.model, test_rows, loaded.train_config, data_cfg, paths)
drop = clean_out.metrics.macro_f1 - ablation.macro_f1
print(f"full frame {clean_out.metrics.macro_f1:.3f}  ->  cropped {ablation.macro_f1:.3f}")
print(f"drop {drop:.3f} —", "reading the leaf" if drop < 0.10 else "LIKELY BACKGROUND SHORTCUT")

In [ ]:
report = comparison_report(
    clean_out.metrics, original_out.metrics,
    ablation=ablation, calibration=calibration.to_dict(),
)
write_evaluation(report, {
    "clean": clean_out.metrics.to_dict(),
    "original": original_out.metrics.to_dict(),
    "ablation": ablation.to_dict(),
    "calibration": calibration.to_dict(),
}, paths)

from IPython.display import Markdown, display
display(Markdown(report))

## 8. See what the model looks at

Heat on lesions and leaf tissue means it works. Heat on sky, soil, or a hand means the score
is borrowed from context. Files starting `WRONG_` are the mistakes — where shortcuts show up
most clearly.

In [ ]:
from banana_lifeline.vision.explain import generate_overlays
import matplotlib.pyplot as plt
from PIL import Image

overlay_dir = generate_overlays(loaded.model, test_rows, loaded.train_config, data_cfg, paths)
shots = sorted(overlay_dir.glob("*.jpg"))[:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axes.flatten(), shots):
    ax.imshow(Image.open(p)); ax.set_title(p.stem.replace("_", " ")[:44], fontsize=8); ax.axis("off")
for ax in axes.flatten()[len(shots):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 9. Get the results off the machine

Only needed if the kernel is on a Colab VM — that storage is wiped when the runtime ends.

In [ ]:
if IN_COLAB_VM:
    DEST = pathlib.Path("/content/drive/MyDrive/banana_runs")
    DEST.mkdir(parents=True, exist_ok=True)
    for part in ("runs", "reports"):
        src = AI_ROOT / part
        if src.is_dir():
            shutil.copytree(src, DEST / part, dirs_exist_ok=True)
    print("saved to", DEST)
else:
    print("Kernel is local — results already on your disk:")
    print(" ", AI_ROOT / "runs")
    print(" ", AI_ROOT / "reports")

---

## Reading the results

**Report the honest numbers.** The contaminated ones exist to show what the broken split
would have let you claim; the gap between them is the finding, not an embarrassment.

- **Panama recall** — smallest class, most costly to miss. Weak here is an honest finding
  about the data and an argument for more Panama photos, not for a different model.
- **The confusion matrix** — Panama/Sigatoka confusion is expected; 16 of 18 contested
  photos were ruled impossible to separate by review.
- **The ablation drop** — large means the model leans on background and will not transfer.